# First experiments with whole Project architecture setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from constraints.lightning_wrappers.modules import ProjectLightning
from constraints.datatools.datasets import CachedArtificalDataset
from constraints import get_experiment_folder, get_data_folder, show_torch_image
from constraints.transforms.transformers import RigidTransformer
from constraints.computers.loss_computers import ProjectLossComputer
from constraints.losses import OneSideSDFSquare
from constraints.models.affine import ProjectWithTemplateA 
from constraints.computers.loss_computers import CrossEntrAndOneSide

from pathlib import Path

import torch
import pytorch_lightning as pl
FODLER = get_experiment_folder(Path("ex3")/"project_debug")
DATA =get_data_folder() / "artificial" / "downloaded"
TRN_FOLDER = DATA / "trn" / "affine"
VAL_FOLDER = DATA / "val" / "affine"

/mnt/appl/software/protobuf-python/6.31.1-GCCcore-14.2.0/lib/python3.13/site-packages/google/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


In [3]:
WANDB_PROJECT = "Constraints"
WANDB_ENTITY = "ksicht"
print(f"Experiment folder: {FODLER}")
print(f"W&B project: {WANDB_ENTITY}/{WANDB_PROJECT}")

Experiment folder: /mnt/personal/mrkosmic/synced/constraints/outputs/notebooks/ex3/project_debug
W&B project: ksicht/Constraints


In [4]:
trn_dataset = CachedArtificalDataset(TRN_FOLDER, sdf_mode="scipy")
val_dataset = CachedArtificalDataset(VAL_FOLDER, sdf_mode="scipy")

In [5]:
from constraints.computers.metric_computers import LabelTripletImageMetricComputer

In [6]:
transformer = RigidTransformer()
loss_computer = CrossEntrAndOneSide()
net = ProjectWithTemplateA(max_translation=0.5)
metric_computer = LabelTripletImageMetricComputer(
    stage="val",
    every_n_epochs=1,
    sample_idx=0,
    image_tag="labels_overlay",
)
module = ProjectLightning(
    net,
    transformer,
    loss_computer,
    metric_computer=metric_computer,
)

In [9]:
from torch.utils.data import DataLoader
from pytorch_lightning.loggers import WandbLogger
# from pytorch_lightning.callbacks import TQDMProgressBar
import wandb

BATCH_SIZE = 32
NUM_WORKERS = 4
EPOCHS = 40
LR = 1e-3

trn_loader = DataLoader(
    trn_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

# ProjectLightning in this repo does not define configure_optimizers, so wire it here.
module.configure_optimizers = lambda: torch.optim.Adam(module.parameters(), lr=LR)

wandb_logger = WandbLogger(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name="ex3-project-debug",
    tags=["scratch", "overlay", "ex3"],
    settings=wandb.Settings(console="wrap"),  # pass settings through here instead
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    devices="auto",
    logger=wandb_logger,
    log_every_n_steps=1,
    enable_checkpointing=False,
    enable_progress_bar=True,
    callbacks=[],
)

trainer.fit(module, train_dataloaders=trn_loader, val_dataloaders=val_loader)

wandb_logger.experiment.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mrkosmic/.netrc.
wandb: Currently logged in as: majkl-mrkos (ksicht) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/mnt/appl/software/PyTorch-Lightning/2.5.5-foss-2025a-CUDA-12.8.0/lib/python3.13/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:493: The total number of parameters detected may be inaccurate because the model contains an instance of `UninitializedParameter`. To get an accurate number, set `self.example_input_array` in your LightningModule.

  | Name              | Type                            | Params | Mode 
------------------------------------------------------------------------------
0 | model             | ProjectWithTemplateA            | 35.6 M | train
1 | spatial_transform | RigidTransformer                | 0      | train
2 | loss_computer     | CrossEntrAndOneSide             | 0      | train
3 | metric_computer   | LabelTripletImageMetricComputer | 0      | train
------------------------------------------------------------------------------
35.6 M    Trainable params
0         Non-trainable params
35.6 M  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=40` reached.


epoch,▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇██
train/iou/pred_vs_gt_epoch,▁▂▃▅▅▆▆▆█▇▇▄▃▃▄▄▄▅▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆
train/iou/pred_vs_gt_step,▁▂▂▃▃▅▇▇▇█▇▆▆▆▃▃▄▄▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▅
train/iou/warped_vs_gt_epoch,▁▃▃▃▃▃▃▃▄▄▄▄▄▅▅▆▆▇▇▇▇▇▇▇█▇██████████████
train/iou/warped_vs_gt_step,▁▄▅▅▄▄▄▅▅▅▅▅▅▄▅▆▆▇▇▇▆▇▆▇▇▇█▇▇█▇██▇▇██▇██
train/loss/loss_sdf_epoch,█▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss/loss_sdf_step,█▇▆▆▄▄▄▅▄▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss/loss_seg_epoch,█▅▃▂▂▁▁▁▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss/loss_seg_step,█▄▃▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss_epoch,█▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+13,...


In [ ]:
from constraints.types import MetricInput

val_batch = next(iter(val_loader))
segmentation_logits, warp_result = module.forward(val_batch["image"], val_batch["template"])

preview_input = MetricInput(
    stage="val",
    batch_idx=0,
    current_epoch=0,
    global_step=0,
    image=val_batch["image"],
    segmentation_logits=segmentation_logits,
    warped_template=warp_result.warped_template,
    gt_mask=val_batch["mask"],
    gt_mask_sdf=val_batch["sdf"],
    transform_spec=warp_result.transform_spec,
)
preview_metric_output = metric_computer.compute(preview_input)
list((preview_metric_output.wandb_overlays or {}).keys())

In [ ]:
from constraints.visu.helpers import show_torch_image
show_torch_image(val_batch["image"][0], title="Input image")
show_torch_image(val_batch["template"][0], title="Template")

In [ ]:
overlay = (preview_metric_output.wandb_overlays or {}).get("labels_overlay")
if overlay is None:
    print("No overlay produced (check stage/epoch/batch gating).")
else:
    show_torch_image(overlay.image, title="Overlay background")
    for mask_name, mask_tensor in overlay.masks.items():
        show_torch_image(mask_tensor, title=f"Mask: {mask_name}", cmap="viridis")